In [1]:
# تثبيت vLLM المتوافق مع إصدار PyTorch في بيئة Colab الحالية
!pip install --no-cache-dir vllm

In [5]:
# 1. إعداد بيئة Python 3.10 لتوافق vLLM و CUDA
!apt-get update -qq && apt-get install -qq python3.10 python3.10-distutils python3.10-venv -y
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

import subprocess, sys, os, signal, time, urllib.request, urllib.error

PY310 = "/usr/bin/python3.10"
PORT = 8000
SERVER_LOG = "/content/server.log"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

# Pins المعتمدة
VLLM_PIN = "0.6.*"
AUTOAWQ_PIN = "0.2.9"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

# 2. تثبيت الحزم داخل بايثون 3.10
cmd_install = [
    PY310, "-m", "pip", "install",
    f"vllm=={VLLM_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}"
]

print("Installing pins on Python 3.10 (silent install, please wait)...")
subprocess.run(cmd_install, check=True)
print("Pins installed successfully.")

# 3. إعداد الرايات وإطلاق الخادم
cmd_server = [
    PY310, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--dtype", "half",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--port", str(PORT),
    "--quantization", "awq",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes"
]

# إنهاء أي عملية قديمة
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
time.sleep(2)

print("Launching server:", " ".join(cmd_server))
logf = open(SERVER_LOG, "wb")
server = subprocess.Popen(cmd_server, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
print(f"Server PID {server.pid}, logging to {SERVER_LOG}")

# 4. فحص الجاهزية (Health Poll)
def wait_for_health(port=PORT, timeout_s=400, interval_s=4):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    print("Waiting for server to report healthy...")
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"Server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("Last 30 log lines:")
    try:
        with open(SERVER_LOG, "r", errors="replace") as fh:
            print("".join(fh.readlines()[-30:]))
    except FileNotFoundError:
        print("(no log file)")
    return False

healthy = wait_for_health()
assert healthy, "Server failed to reach healthy state."

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../0-libpython3.10-dev_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../1-libpython3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../2-python3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking python3.10 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../3-libpython3.10-stdlib_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-stdlib:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../4-python3.10-minimal_3.10.12-1~22.04.17_amd64.deb ...
Unpacking python3.10-minimal (3.10

In [6]:
# 1. كتابة ملف prompts.txt بالأسئلة العشرين المحددة في معايير اللاب
prompts_content = """What is a GPU?
Define tokens per second in one line.
Explain the difference between prefill and decode in two sentences.
List three reasons decode is memory-bound rather than compute-bound.
Summarise what an inference server does for an ops team, in three short bullets.
Why does a longer prompt increase time to first token but not the per-token gap?
Describe the KV cache to a new engineer and say why it grows with context length.
Walk through what continuous batching changes versus static batching, with an example of the straggler effect it removes.
Name two things weight-only quantisation trades away in exchange for smaller memory footprint.
A user asks for the weather in Riyadh and the current time in Tokyo; describe the two tool calls you would make and the arguments for each.
Write a short runbook for rolling back a bad deployment, listing the steps in order and the check after each one.
Explain, for a non-technical manager, why a busy GPU is not the same as a productive GPU, using the utilisation trap.
Compare fp16 and int4 for serving a 1.5 billion parameter model: memory, speed, and quality, in a short paragraph each.
Give a one-sentence definition of p95 latency and say why it matters more than the average for an SLO.
Draft three sentences a platform team could send another team to describe an OpenAI-compatible endpoint they can call.
Outline the symptom, hypothesis, and measurement steps you would take when throughput is lower than expected under load.
What is PagedAttention and what problem in KV cache memory does it solve? Answer in two sentences.
Explain why the knee at the SLO, not the peak throughput, is the honest capacity number for a benchmark.
Describe how you would size the GPU memory budget for a model plus its KV cache before ever loading it.
Write a calm status update for a channel of engineers explaining that latency has risen, what you suspect, and what you are doing about it, in four sentences."""

with open("prompts.txt", "w", encoding="utf-8") as f:
    f.write(prompts_content.strip())
print("prompts.txt created successfully.")

# 2. كتابة سكربت bench.py المتوافق تماماً مع متطلبات اللاب وعقد JSON
bench_code = '''import argparse
import json
import os
import sys
import time
import urllib.request
import urllib.error
import numpy as np
from concurrent.futures import ThreadPoolExecutor

def send_request(base_url, model, prompt):
    url = f"{base_url}/v1/completions"
    payload = json.dumps({
        "model": model,
        "prompt": prompt,
        "max_tokens": 128,
        "temperature": 0.0,
        "stream": True
    }).encode("utf-8")

    req = urllib.request.Request(url, data=payload, headers={"Content-Type": "application/json"})
    t0 = time.time()
    ttft = None
    tokens = 0

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            for line in resp:
                line = line.decode("utf-8").strip()
                if line.startswith("data: ") and line != "data: [DONE]":
                    data = json.loads(line[6:])
                    if ttft is None:
                        ttft = time.time() - t0
                    tokens += len(data.get("choices", [{}])[0].get("text", ""))
        lat = time.time() - t0
        return True, ttft if ttft else lat, lat, tokens
    except Exception as e:
        return False, 0.0, 0.0, 0

def run_level(base_url, model, concurrency, req_count, prompts):
    # مرحلة إحماء خفيفة للمستوى لا تدخل في الحساب
    warm_prompt = prompts[0]
    send_request(base_url, model, warm_prompt)

    prompts_to_use = [prompts[i % len(prompts)] for i in range(req_count)]

    t_start = time.time()
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        results = list(executor.map(lambda p: send_request(base_url, model, p), prompts_to_use))
    total_time = time.time() - t_start

    successes = [r for r in results if r[0]]
    errors = len(results) - len(successes)

    if not successes:
        return {
            "concurrency": concurrency,
            "tokens_per_s": 0.0,
            "ttft_p50_s": 0.0,
            "ttft_p95_s": 0.0,
            "latency_p95_s": 0.0,
            "errors": errors
        }

    ttfts = [r[1] for r in successes]
    lats = [r[2] for r in successes]
    total_tokens = sum(r[3] for r in successes)

    return {
        "concurrency": concurrency,
        "tokens_per_s": round(total_tokens / total_time, 2),
        "ttft_p50_s": round(float(np.percentile(ttfts, 50)), 3),
        "ttft_p95_s": round(float(np.percentile(ttfts, 95)), 3),
        "latency_p95_s": round(float(np.percentile(lats, 95)), 3),
        "errors": errors
    }

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base-url", required=True)
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", required=True)
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", required=True)
    parser.add_argument("--out", required=True)
    args = parser.parse_args()

    with open(args.prompt_file) as f:
        prompts = [line.strip() for line in f if line.strip()]

    concurrency_levels = [int(c.strip()) for c in args.concurrency.split(",")]

    report_doc = {"runs": [{"levels": []}]}
    if os.path.exists(args.out):
        try:
            with open(args.out) as f:
                report_doc = json.load(f)
        except Exception:
            pass

    current_levels = []
    print(f"Starting sweep for model: {args.model}")
    for c in concurrency_levels:
        res = run_level(args.base_url, args.model, c, args.requests_per_level, prompts)
        current_levels.append(res)
        print(f"c={res['concurrency']:>2}  tok/s={res['tokens_per_s']:>7.1f}  "
              f"ttft_p95={res['ttft_p95_s']:.3f}s  lat_p95={res['latency_p95_s']:.3f}s  "
              f"errors={res['errors']}")

        # حفظ فوري لتفادي فقدان البيانات عند حدوث انقطاع
        report_doc["runs"][-1]["levels"] = current_levels
        with open(args.out, "w") as f:
            json.dump(report_doc, f, indent=2)

if __name__ == "__main__":
    main()
'''

with open("bench.py", "w", encoding="utf-8") as f:
    f.write(bench_code)
print("bench.py generated.")

prompts.txt created successfully.
bench.py generated.


In [7]:
!/usr/bin/python3.10 bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

Starting sweep for model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
c= 1  tok/s=  462.1  ttft_p95=0.068s  lat_p95=1.419s  errors=0
c= 2  tok/s=  858.8  ttft_p95=0.167s  lat_p95=1.601s  errors=0
c= 4  tok/s= 1527.1  ttft_p95=0.187s  lat_p95=1.737s  errors=0
c= 8  tok/s= 2357.0  ttft_p95=0.128s  lat_p95=1.804s  errors=0
c=16  tok/s= 3142.4  ttft_p95=0.162s  lat_p95=2.267s  errors=0


In [8]:
import json

# قراءة النتائج
with open("bench_report.json") as f:
    report = json.load(f)

levels = report["runs"][-1]["levels"]

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}  lat_p95={L['latency_p95_s']:.3f}  "
          f"errors={L['errors']}")

# تحديد الـ SLO المستهدف
TARGET_P95_S = 2.5

# استخراج أعلى مستوى تزامن بقي تحت الهدف
under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None
print("\nSelected Knee:", knee)

# كتابة ملف knee.json للتحقق الأخضر (Green Check)
with open("knee.json", "w") as f:
    json.dump({
        "target_p95_s": TARGET_P95_S,
        "knee_concurrency": knee["concurrency"] if knee else None
    }, f, indent=2)

print("knee.json written successfully.")

c= 1  tok/s=  462.1  ttft_p95=0.068  lat_p95=1.419  errors=0
c= 2  tok/s=  858.8  ttft_p95=0.167  lat_p95=1.601  errors=0
c= 4  tok/s= 1527.1  ttft_p95=0.187  lat_p95=1.737  errors=0
c= 8  tok/s= 2357.0  ttft_p95=0.128  lat_p95=1.804  errors=0
c=16  tok/s= 3142.4  ttft_p95=0.162  lat_p95=2.267  errors=0

Selected Knee: {'concurrency': 16, 'tokens_per_s': 3142.41, 'ttft_p50_s': 0.154, 'ttft_p95_s': 0.162, 'latency_p95_s': 2.267, 'errors': 0}
knee.json written successfully.


In [9]:
capacity_note_content = """# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency (your SLO today): 2.5 seconds
- Knee concurrency (highest concurrency whose p95 is still under target): 16
- Tokens per second at the knee: 3142.4
- Max sustainable request rate at the target p95: 7.06 req/s

## The limiting family

One sentence, using this morning's triage lens (compute vs memory vs overhead):
which family limits this stack at the knee, and the tell that points to it.

- Memory-bound: token throughput begins tapering off its linear scaling slope while decode memory bandwidth and KV cache management dominate execution time rather than raw compute.

## Why the knee, not the peak

One sentence in your own words on why you report the knee at the SLO rather than
the peak throughput.

- Reporting the knee reflects the honest usable serving capacity under our latency contract, whereas peak throughput includes saturated requests that fail acceptable latency limits.
"""

with open("capacity-note.md", "w", encoding="utf-8") as f:
    f.write(capacity_note_content.strip())

print("capacity-note.md generated successfully.")

capacity-note.md generated successfully.


In [10]:
# Green-check verifier for Lab W3D5
import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}

class _Stop(Exception):
    pass

def fail(reason: str):
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()

def main():
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found")
    with open("bench_report.json") as fh:
        document = json.load(fh)

    levels = document["runs"][-1].get("levels") if isinstance(document, dict) and "runs" in document else document
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        total_errors += L["errors"]

    if not os.path.exists("knee.json"):
        fail("knee.json not found")
    with open("knee.json") as fh:
        knee = json.load(fh)
    if knee.get("target_p95_s", 0) <= 0:
        fail("target_p95_s must be positive")
    if knee.get("knee_concurrency", 0) < 1:
        fail("knee_concurrency must be valid")

    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    if re.findall(r"FILL:", note):
        fail("unfilled FILL: placeholders found in capacity-note.md")

    print(f"levels: {len(levels)}, concurrencies: {[l['concurrency'] for l in levels]}, total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")

try:
    main()
except _Stop:
    pass

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [11]:
# 1. إيقاف الخادم وتنظيف المنفذ
import os, signal, subprocess, time

subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
time.sleep(2)
print("Server stopped cleanly.")

# 2. تحميل الملفات قبل إغلاق الجلسة
from google.colab import files
for f_ in ["bench_report.json", "capacity-note.md", "knee.json"]:
    files.download(f_)

Server stopped cleanly.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>